In [ ]:
{
  "id": "z_7SqZ0BEUL25CGNd9nP",
  "name": "zabbix-ai-agent-api",
  "api_key": "ZhSUGqVf4r1oBLHxr4Otlg",
  "encoded": "el83U3FaMEJFVUwyNUNHTmQ5blA6WmhTVUdxVmY0cjFvQkxIeHI0T3RsZw=="
}

Get Cluster info

In [26]:
from elasticsearch import Elasticsearch
import json

es_endpoint = "https://10.120.115.183:9200"
# https://10.120.115.183:9200/_cat/indices?v


#es_endpoint = "https://observability.dev.e2open.com/logs/zabbix_api/_search" 
es_api_key =  "el83U3FaMEJFVUwyNUNHTmQ5blA6WmhTVUdxVmY0cjFvQkxIeHI0T3RsZw=="


es = Elasticsearch(
    es_endpoint,
    api_key=es_api_key,
    verify_certs=False
)

# Test connection
print(es.info())

c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'name': 'dev5232-hot', 'cluster_name': 'observability-dev-int', 'cluster_uuid': 'LE14Lp--TK2HgKp5YoXXhA', 'version': {'number': '9.3.1', 'build_flavor': 'default', 'build_type': 'rpm', 'build_hash': '0dd66e52ba3aa076cf498264e46339dbb71f0269', 'build_date': '2026-02-23T23:37:38.684779921Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


{'name': 'dev5232-hot', 'cluster_name': 'observability-dev-int', 'cluster_uuid': 'LE14Lp--TK2HgKp5YoXXhA', 'version': {'number': '9.3.1', 'build_flavor': 'default', 'build_type': 'rpm', 'build_hash': '0dd66e52ba3aa076cf498264e46339dbb71f0269', 'build_date': '2026-02-23T23:37:38.684779921Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}

create index in zabbix_api on elasticsearch

In [2]:
index_name = "zabbix_api"

mapping = {
    "mappings": {
        "properties": {
            "method": {"type": "keyword"},
            "object_type": {"type": "keyword"},
            "content": {"type": "text"},
            "embedding": {
                "type": "dense_vector",
                "dims": 384
            },
            "metadata": {"type": "object"}
            
        }
    }
}


# reset index
#if es.indices.exists(index=index_name):
#    es.indices.delete(index=index_name)

es.indices.create(index=index_name, body=mapping)

print("✅ Index created")

c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


BadRequestError: BadRequestError(400, 'resource_already_exists_exception', 'index [zabbix_api/aBKOBtcBTVm41V9CSSQ5EQ] already exists')

In [3]:
from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer




index_name = "zabbix_api"

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embedding(text):
    return model.encode(text).tolist()

# Your documents (from your chunks)
docs =[
  {
    "object_type": "host",
    "method": "host.get",
    "content": """ # Host API  ## host.get  Description: Retrieve host information from Zabbix. This method supports filtering hosts and retrieving related data such as interfaces, items, triggers, groups, templates, and inventory.  Common Use Cases: - Get host details by name - Retrieve network interfaces of a host - Fetch host items or triggers - Get hosts by group, template, or tags - Filter hosts based on problem severity  Method: host.get  Required Fields: - method: host.get - params: object  Important Parameters:  Filtering: - filter: Filter hosts by exact values (e.g., host name) - hostids: Retrieve specific hosts by ID - groupids: Filter by host groups - templateids: Filter by linked templates - proxyids: Filter hosts monitored by proxies  Search: - search: Pattern-based search (LIKE) - searchInventory: Search using inventory fields  Host State: - monitored_hosts: Return only monitored hosts - withProblemsSuppressed: Return hosts with suppressed problems - severities: Filter hosts by problem severity  Tags: - tags: Filter hosts using tag conditions - inheritedTags: Include template-level tags - evaltype: Tag evaluation logic (AND/OR)  Data Expansion (VERY IMPORTANT): - selectInterfaces: Retrieve network interfaces - selectItems: Retrieve host items - selectTriggers: Retrieve triggers - selectHostGroups: Retrieve host groups - selectParentTemplates: Retrieve linked templates - selectInventory: Retrieve inventory data - selectTags: Retrieve host tags  Output Control: - output: Specify fields to return (use "extend" for all fields) - limit: Limit number of results - sortfield: Sort by hostid, host, name, status  Intent Mapping (IMPORTANT FOR AI):  If user asks about: - "IP", "interface", "network" → use selectInterfaces - "items", "metrics" → use selectItems - "triggers", "alerts" → use selectTriggers - "groups" → use selectHostGroups - "templates" → use selectParentTemplates - "inventory" → use selectInventory - "tags" → use selectTags  If hostname is mentioned: → Use filter.host  Example Request (Basic Host Lookup): {     "jsonrpc": "2.0",     "method": "host.get",     "params": {         "filter": {             "host": ["Zabbixserver", "Linuxserver"]         }     },     "id": 1 }  Example Request (Get Interfaces): {     "jsonrpc": "2.0",     "method": "host.get",     "params": {         "output": ["hostid", "host"],         "filter": {             "host": ["dev444"]         },         "selectInterfaces": "extend"     },     "id": 1 }  Example Request (Get Host Items): {     "jsonrpc": "2.0",     "method": "host.get",     "params": {         "output": ["hostid"],         "selectItems": ["itemid", "name", "status"],         "filter": {             "host": ["Zabbixserver"]         }     },     "id": 1 }  Example Request (Filter by Tags): {     "jsonrpc": "2.0",     "method": "host.get",     "params": {         "output": ["hostid"],         "selectTags": "extend",         "tags": [             {                 "tag": "host-name",                 "value": "linux-server",                 "operator": 1             }         ]     },     "id": 1 }  Example Response: {     "jsonrpc": "2.0",     "result": [         {             "hostid": "10160",             "host": "Zabbixserver",             "status": "0"         }     ],     "id": 1 }  Notes: - Use "output": "extend" to get all fields - Use "select*" fields to include related data - Use "filter" for exact matching - Use "search" for partial matching - Combine filters for precise results """
  },
  {
    "object_type": "problem",
    "method": "problem.get",
    "content": """ # Problem API  ## problem.get  Description: Retrieve problem events from Zabbix. This method returns unresolved problems and optionally recently resolved problems depending on configuration. It is mainly used to monitor active issues, alerts, and incidents in the system.  Common Use Cases: - Get active problems - Retrieve recent alerts - Filter problems by severity - Get problems for a specific host - Retrieve acknowledged or suppressed problems - Analyze alert history (recent)  Method: problem.get  Required Fields: - method: problem.get - params: object  Important Parameters:  Filtering: - eventids: Retrieve specific problem events - hostids: Filter problems by host - groupids: Filter problems by host group - objectids: Filter by trigger/object ID  Problem State: - acknowledged: true/false → filter acknowledged problems - suppressed: true/false → filter suppressed problems - symptom: true → return symptom events  Severity: - severities: Filter problems by severity levels (0–5)  Time Filters: - time_from: Start time (timestamp) - time_till: End time (timestamp) - recent: true → include recently resolved problems  Tag Filtering: - tags: Filter using tag conditions - evaltype: Tag evaluation logic (AND/OR)  Event Actions: - action: Filter by event actions (acknowledge, suppress, etc.) - action_userids: Filter by users who performed actions  Data Expansion (VERY IMPORTANT): - selectAcknowledges: Retrieve acknowledgment history - selectTags: Retrieve problem tags - selectSuppressionData: Retrieve suppression/maintenance info  Output Control: - output: Fields to return ("extend" recommended) - limit: Limit number of results - sortfield: Usually "eventid" - sortorder: "ASC" or "DESC"  Intent Mapping (IMPORTANT FOR AI):  If user asks about: - "activeproblems", "currentissues" → use problem.get - "alerts", "incidents" → use problem.get - "severity", "critical", "high" → use severities - "hostproblems" → use hostids - "acknowledgedproblems" → use acknowledged=true - "suppressedproblems" → use suppressed=true - "recentissues" → use recent=true - "problemhistory(recent)" → use recent=true  Example Request (Get Active Problems): {     "jsonrpc": "2.0",     "method": "problem.get",     "params": {         "output": "extend",         "recent": true,         "sortfield": ["eventid"],         "sortorder": "DESC"     },     "id": 1 }  Example Request (Filter by Severity): {     "jsonrpc": "2.0",     "method": "problem.get",     "params": {         "output": "extend",         "severities": [4, 5],         "recent": true     },     "id": 1 }  Example Request (Problems for Host): {     "jsonrpc": "2.0",     "method": "problem.get",     "params": {         "output": "extend",         "hostids": ["12345"],         "recent": true     },     "id": 1 }  Example Request (With Details): {     "jsonrpc": "2.0",     "method": "problem.get",     "params": {         "output": "extend",         "selectAcknowledges": "extend",         "selectTags": "extend",         "selectSuppressionData": "extend",         "recent": true     },     "id": 1 }  Example Response: {     "jsonrpc": "2.0",     "result": [         {             "eventid": "1245463",             "name": "Zabbixagentunreachable",             "severity": "3",             "acknowledged": "1",             "suppressed": "1"         }     ],     "id": 1 }  Notes: - By default, only unresolved problems are returned - Use "recent": true to include recently resolved problems - Use "select*" fields to get additional data - Combine filters for precise results - Use event.get for historical (older) problems """
  }
]

# Insert documents
for doc in docs:
    embedding = get_embedding(doc["content"])

    body = {
        "method": doc["method"],
        "object_type": doc["object_type"],
        "content": doc["content"],
        "embedding": embedding
       
    }

    es.index(index=index_name, document=body)

print("✅ Documents inserted")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ Documents inserted


In [13]:
#insert hostgroup.get 

from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer

# ================================
# hostgroup.get (SINGLE DOCUMENT)
# ================================

hostgroup_content = """
# Host Group API

## hostgroup.get

Description:
Retrieve host group information from Zabbix. This method allows filtering host groups and retrieving related hosts, items, triggers, and discovery data.

Common Use Cases:
- Get all host groups
- Filter host groups by name or ID
- Get host groups for specific hosts
- Retrieve hosts inside a group
- Get monitored host groups

Method:
hostgroup.get

Required Fields:
- method: hostgroup.get
- params: object

Filtering Parameters:

- groupids: Retrieve specific host groups by ID
- hostids: Return groups that contain given hosts
- graphids: Return groups with hosts having graphs
- triggerids: Return groups with hosts having triggers
- maintenanceids: Return groups affected by maintenance

Boolean Filters:

- with_hosts: Only groups that contain hosts
- with_items: Only groups with items
- with_triggers: Only groups with triggers
- with_graphs: Only groups with graphs
- with_httptests: Only groups with web checks

Monitored Filters:

- with_monitored_hosts: Only monitored hosts
- with_monitored_items: Only enabled items
- with_monitored_triggers: Only enabled triggers
- with_monitored_httptests: Only enabled web checks


Search and Output Control:

- filter: Exact match filtering (e.g., name)
- search: Pattern-based search
- output: Fields to return ("extend" recommended)
- limit: Limit number of results
- sortfield: Sort by groupid or name
- sortorder: ASC or DESC
- countOutput: Return count instead of full data

Other Options:

- editable: Return only editable groups
- preservekeys: Keep original keys in response

Data Expansion (VERY IMPORTANT):

- selectHosts: Retrieve hosts in the group
- selectDiscoveryRules: Get discovery rules
- selectDiscoveryData: Get discovery details
- selectHostPrototypes: Get host prototypes

Notes:
- selectHosts supports count
- limitSelects can limit subquery results

Discovery Data Details:

Each discovery object contains:

- parent_group_prototypeid: Prototype ID
- name: Prototype name
- ts_delete: Deletion timestamp
- status:
    0 → discovered
    1 → no longer discovered

Note:
- selectGroupDiscoveries is deprecated
- Use selectDiscoveryData instead

Intent Mapping (IMPORTANT FOR AI):

If user asks about:

- "host groups", "groups" → use hostgroup.get
- "hosts in group" → use selectHosts
- "group by name" → use filter.name
- "monitored groups" → use with_monitored_hosts
- "groups with problems/triggers" → use with_triggers
- "groups with items" → use with_items

Example Request (Basic):

{
    "jsonrpc": "2.0",
    "method": "hostgroup.get",
    "params": {
        "output": "extend"
    },
    "id": 1
}

Example Request (Filter by Name):

{
    "jsonrpc": "2.0",
    "method": "hostgroup.get",
    "params": {
        "output": "extend",
        "filter": {
            "name": ["Zabbix servers", "Linux servers"]
        }
    },
    "id": 1
}

Example Response:

{
    "jsonrpc": "2.0",
    "result": [
        {
            "groupid": "2",
            "name": "Linux servers"
        },
        {
            "groupid": "4",
            "name": "Zabbix servers"
        }
    ],
    "id": 1
}

"""

# 
# INSERT 
# 

doc = {
    "method": "hostgroup.get",
    "object_type": "hostgroup",
    "content": hostgroup_content,
    "embedding": get_embedding(hostgroup_content),
    "metadata": {}
}

es.index(index="zabbix_api", document=doc)

print("✅ hostgroup.get appended safely")

c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ hostgroup.get appended safely


Listing all index doc


In [17]:
# Equivalent to _cat/indices?v
indices = es.cat.indices(format="json")
for index in indices:
    print(f"Index: {index['index']} | Status: {index['health']} | Docs: {index['docs.count']}")

c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


AuthorizationException: AuthorizationException(403, 'security_exception', 'action [indices:monitor/settings/get] is unauthorized for API key id [z_7SqZ0BEUL25CGNd9nP] of user [Observability_service_account], this action is granted by the index privileges [monitor,view_index_metadata,manage,all]')

Fetch All Documents (Match All)

In [15]:

index_name = "zabbix_api"

response = es.search(
    index=index_name,
    query={"match_all": {}},
    size=3
)

print(f"Total documents found: {response['hits']['total']['value']}")
for hit in response['hits']['hits']:
    print(hit['_source'])

c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Total documents found: 3
{'method': 'host.get', 'object_type': 'host', 'content': ' # Host API  ## host.get  Description: Retrieve host information from Zabbix. This method supports filtering hosts and retrieving related data such as interfaces, items, triggers, groups, templates, and inventory.  Common Use Cases: - Get host details by name - Retrieve network interfaces of a host - Fetch host items or triggers - Get hosts by group, template, or tags - Filter hosts based on problem severity  Method: host.get  Required Fields: - method: host.get - params: object  Important Parameters:  Filtering: - filter: Filter hosts by exact values (e.g., host name) - hostids: Retrieve specific hosts by ID - groupids: Filter by host groups - templateids: Filter by linked templates - proxyids: Filter hosts monitored by proxies  Search: - search: Pattern-based search (LIKE) - searchInventory: Search using inventory fields  Host State: - monitored_hosts: Return only monitored hosts - withProblemsSuppress

See Index Mapping (Structure)

In [19]:
mapping = es.indices.get_mapping(index=index_name)
import json
print(json.dumps(mapping[index_name], indent=2))

c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


AuthorizationException: AuthorizationException(403, 'security_exception', 'action [indices:admin/mappings/get] is unauthorized for API key id [z_7SqZ0BEUL25CGNd9nP] of user [Observability_service_account] on indices [zabbix_api], this action is granted by the index privileges [view_index_metadata,manage,all]')

In [21]:
query1 = "how many host are their in zabbix"

qurey1_vector = get_embedding(query1)
print(f"query1_vector: {qurey1_vector}") 

query2 = "what are the problelms of host dev4506.dev.e2open.com"
query2_vector = get_embedding(query2)
print(f"query2_vector: {query2_vector}")

query1_vector: [0.005023303907364607, 0.019264554604887962, -0.07784843444824219, 0.0055335029028356075, -0.058682672679424286, -0.12350531667470932, 0.029525885358452797, -0.09464342147111893, 0.058367349207401276, -0.024030311033129692, -0.08219827711582184, 0.007033481262624264, 0.1431855410337448, -0.029116470366716385, 0.03934597596526146, -0.10270305722951889, 0.034122973680496216, -0.12820464372634888, 0.02574142813682556, -0.009309510700404644, -0.005344910081475973, -0.01912027783691883, 0.10602803528308868, -0.03547857329249382, -0.05221092700958252, 0.0021002113353461027, 0.018697459250688553, 0.12291325628757477, -0.032567091286182404, -0.06895989179611206, 0.11400756239891052, 0.11852549761533737, -0.05306926742196083, 0.030752407386898994, -0.005635376553982496, 0.007936712354421616, -0.01776842400431633, 0.07497406005859375, -0.08260896801948547, 0.006464492995291948, 0.09964921325445175, 0.017184752970933914, -0.008018520660698414, 0.024998068809509277, -0.0842117294669

Performing  Semamtic search on zabbix_api Index

In [29]:
# query_vector should be the output from your MiniLM or text-embedding-004 model

search_query = {
    "knn": {
        "query": {"match_all": {}},
        "field": "embedding", 
        "query_vector": qurey1_vector,
        "k": 5,
        "num_candidates": 100,
        "filter": {
            "term": { "metadata.object_type": "host" } # Your isolation requirement
        }
    }
}


results = es.search(index=index_name, body=search_query)

c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


BadRequestError: BadRequestError(400, 'x_content_parse_exception', '[1:9] [knn] unknown field [query]')

In [28]:
from elasticsearch import Elasticsearch


index_name = "zabbix_api"
doc_id = "tyMIup0BB_hFvOhY0A4H"   # document ID

response = es.delete(index=index_name, id=doc_id)

print(response)

c:\Users\dishant.totade\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.120.115.183'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'_index': 'zabbix_api', '_id': 'tyMIup0BB_hFvOhY0A4H', '_version': 2, 'result': 'deleted', '_shards': {'total': 2, 'successful': 2, 'failed': 0}, '_seq_no': 3, '_primary_term': 1}
